# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [1]:

import networkx as nx
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

from pysr import PySRRegressor
import numpy as np
import torch.nn as nn
from torch.optim import Adam
from torch_geometric.utils import from_networkx, add_self_loops
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import KFold


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
import os
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter

In [3]:
def parse_parameters(param_path):
    """
    Parse parameters from the given file.
    """
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)  # Execute the file content in a controlled namespace
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])  # Convert W to NumPy array
    return params

def load_matrices(directory, timepoint):
    """
    Load the graph, properties, and state matrices for a specific timepoint.
    Args:
        directory (str): Directory containing the .npy files.
        timepoint (int): Timepoint to load matrices for.
    Returns:
        graph_mat (np.ndarray): Adjacency matrix.
        properties_mat (np.ndarray): Properties matrix.
        state_mat (np.ndarray): State matrix.
    """
    print(f"Loading matrices for timepoint {timepoint} from {directory}...")
    graph_mat = np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy"))
    properties_mat = np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy"))
    state_mat = np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
    return graph_mat, properties_mat, state_mat

def prepare_training_data(graph_mat, state_mat, next_state_mat, properties_mat, global_constants):
    """
    Prepares the dataset for training the GCA model.
    Encodes graph structure, node features, and global constants.
    Args:
        graph_mat (np.ndarray): Adjacency matrix.
        state_mat (np.ndarray): State matrix at current timepoint.
        next_state_mat (np.ndarray): State matrix at next timepoint.
        properties_mat (np.ndarray): Properties matrix (e.g., area, perimeter).
        global_constants (np.ndarray): Global parameters for the model.
    Returns:
        Data: A torch-geometric Data object for training.
    """
    # Step 1: Prepare node features
    n_cells = properties_mat.shape[0]
    cell_types = np.argmax(state_mat, axis=1)  # Get cell types as integers
    node_features = np.hstack((properties_mat, cell_types.reshape(-1, 1)))  # Combine area, perimeter, and cell_type

    # Step 2: Prepare edge list
    edge_indices = np.transpose(np.nonzero(graph_mat))  # Extract edges from adjacency matrix

    # Step 3: Create a Data object
    graph_data = Data(
        x=torch.tensor(node_features, dtype=torch.float),  # Node features
        edge_index=torch.tensor(edge_indices.T, dtype=torch.long),  # Edge indices
        global_features=torch.tensor(global_constants, dtype=torch.float),  # Global constants
        input_state=torch.tensor(state_mat, dtype=torch.float),  # Input state
        target_state=torch.tensor(next_state_mat, dtype=torch.float),  # Target state
    )

    return graph_data

class GCAModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, global_dim):
        """
        GCA Model for predicting state transitions in the graph.
        Args:
            input_dim (int): Dimension of node input features.
            hidden_dim (int): Dimension of hidden layer embeddings.
            output_dim (int): Dimension of output (next state).
            global_dim (int): Dimension of global constants.
        """
        super(GCAModel, self).__init__()

        # Layers for node feature encoding
        self.node_encoder = nn.Linear(input_dim, hidden_dim)

        # Layers for edge updates
        self.edge_model = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Layers for node state updates
        self.node_updater = nn.Sequential(
            nn.Linear(hidden_dim + global_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x, edge_index, global_features):
        """
        Forward pass of the GCA model.
        Args:
            x (Tensor): Node features [num_nodes, input_dim].
            edge_index (Tensor): Edge list [2, num_edges].
            global_features (Tensor): Global constants [global_dim].
        Returns:
            Tensor: Predicted node states [num_nodes, output_dim].
        """
        # Encode node features
        node_hidden = F.relu(self.node_encoder(x))

        # Compute edge interactions
        edge_sources = edge_index[0]
        edge_targets = edge_index[1]
        edge_features = node_hidden[edge_sources] + node_hidden[edge_targets]  # Combine edge ends
        edge_updates = self.edge_model(edge_features)

        # Aggregate edge updates back to nodes
        node_aggregate = scatter(edge_updates, edge_sources, dim=0, reduce="sum")

        # Concatenate global features and perform node updates
        global_expanded = global_features.expand(node_hidden.shape[0], -1)  # Broadcast globals
        combined_features = torch.cat([node_aggregate, global_expanded], dim=1)
        node_updated = self.node_updater(combined_features)

        return node_updated

def inspect_matrix_dimensions(data_dir, timepoints):
    """
    Inspect the dimensions of graph_mat, state_mat, and properties_mat for each timepoint.
    Args:
        data_dir (str): Directory containing the .npy files.
        timepoints (list): List of timepoints to load.
    """
    for t in timepoints:
        graph_mat_path = os.path.join(data_dir, f"{t}_graph_mat.npy")
        state_mat_path = os.path.join(data_dir, f"{t}_state_mat.npy")
        properties_mat_path = os.path.join(data_dir, f"{t}_properties_mat.npy")

        # Load matrices
        graph_mat = np.load(graph_mat_path)
        state_mat = np.load(state_mat_path)
        properties_mat = np.load(properties_mat_path)

        # Print dimensions
        print(f"Timepoint: {t}")
        print(f"  graph_mat dimensions: {graph_mat.shape}")
        print(f"  state_mat dimensions: {state_mat.shape}")
        print(f"  properties_mat dimensions: {properties_mat.shape}")
        print("-" * 40)





In [12]:
def weighted_mse_loss(predicted, target, weights):
    """
    Compute the weighted mean squared error loss.
    Args:
        predicted (Tensor): Predicted values.
        target (Tensor): Ground truth values.
        weights (Tensor): Weights for each sample.
    Returns:
        Tensor: Weighted MSE loss.
    """
    loss = weights * (predicted - target) ** 2
    return loss.mean()

def train_gca_model(training_data, model, optimizer, num_epochs):
    model.train()
    training_losses = []

    for epoch in range(num_epochs):
        total_loss = 0

        for graph_data in training_data:
            x = graph_data.x
            edge_index = graph_data.edge_index
            global_features = graph_data.global_features
            input_state = graph_data.input_state
            target_state = graph_data.target_state

            # Forward pass
            predicted_state = model(x, edge_index, global_features)

            # Compute weights based on cell type
            cell_types = torch.argmax(input_state, dim=1)
            weights = torch.where(cell_types == 1, torch.tensor(1000.0), torch.tensor(1.0))

            # Compute weighted loss
            loss = weighted_mse_loss(predicted_state, target_state, weights)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(training_data)
        training_losses.append(avg_loss)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}")

    return training_losses

In [ ]:
def inspect_matrix_dimensions(data_dir, timepoints, param_files):
    """
    Inspect the dimensions of graph_mat, state_mat, and properties_mat for each timepoint.
    Args:
        data_dir (str): Directory containing the .npy files.
        timepoints (list): List of timepoints to load.
        param_files (list): List of parameter file paths.
    """
    params_dict = {i: parse_parameters(param_file) for i, param_file in enumerate(param_files)}

    for t_idx in range(len(timepoints) - 1):  # Iterate over pairs of timepoints
        t = timepoints[t_idx]
        next_t = timepoints[t_idx + 1]

        graph_mat = np.load(os.path.join(data_dir, f"{t}_graph_mat.npy"))
        state_mat = np.load(os.path.join(data_dir, f"{t}_state_mat.npy"))
        next_state_mat = np.load(os.path.join(data_dir, f"{next_t}_state_mat.npy"))
        properties_mat = np.load(os.path.join(data_dir, f"{t}_properties_mat.npy"))

        print(f"Timepoint: {t} -> {next_t}")
        print(f"  graph_mat dimensions: {graph_mat.shape}")
        print(f"  state_mat dimensions: {state_mat.shape}")
        print(f"  next_state_mat dimensions: {next_state_mat.shape}")
        print(f"  properties_mat dimensions: {properties_mat.shape}")
        print("-" * 40)

def main(data_dirs, param_files, num_epochs=2, batch_size=32):
    """
    Main workflow for training and validating the GCA model.
    Args:
        data_dirs (list): List of directories containing matrix files for timepoints.
        param_files (list): List of parameter file paths.
        num_epochs (int): Number of training epochs.
        batch_size (int): Batch size for DataLoader.
    Returns:
        model: Trained GCA model.
        training_losses: List of training losses per epoch.
        avg_val_loss: Average validation loss.
    """
    # Initialize global constants for all parameter files
    training_data = []
    validation_data = []

    for idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        # Parse parameters for this directory
        params = parse_parameters(param_file)
        global_constants = np.concatenate([
            np.array(params["v0"]),
            np.array([params["Dr"], params["kappa_A"], params["kappa_P"], params["a"], params["k"]]),
            params["W"].flatten(),
        ])

        # Define timepoints for this directory
        timepoints = list(range(0, 2001, 100))  # Use 6 initial timepoints

        # Inspect dimensions of matrices
        inspect_matrix_dimensions(data_dir, timepoints, param_files)

        # Load data for training or validation based on index
        for t_idx in range(len(timepoints) - 1):
            t = timepoints[t_idx]
            next_t = timepoints[t_idx + 1]

            graph_mat = np.load(os.path.join(data_dir, f"{t}_graph_mat.npy"))
            state_mat = np.load(os.path.join(data_dir, f"{t}_state_mat.npy"))
            next_state_mat = np.load(os.path.join(data_dir, f"{next_t}_state_mat.npy"))
            properties_mat = np.load(os.path.join(data_dir, f"{t}_properties_mat.npy"))

            # Prepare graph data for the current transition
            graph_data = prepare_training_data(graph_mat, state_mat, next_state_mat, properties_mat, global_constants)

            # Append to training or validation set
            if idx < 8:  # First 8 directories for training
                training_data.append(graph_data)
            else:  # Last 2 directories for validation
                validation_data.append(graph_data)

    # Use PyTorch Geometric's DataLoader for batching
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)

    # Step 3: Initialize model, optimizer, and loss function
    input_dim = properties_mat.shape[1] + 1  # area + perimeter + cell_type
    hidden_dim = 64  # Example hidden dimension
    output_dim = state_mat.shape[1]  # Number of cell states
    global_dim = len(global_constants)  # Dimension of global constants

    model = GCAModel(input_dim, hidden_dim, output_dim, global_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = torch.nn.MSELoss()  # Mean Squared Error Loss

    # Step 4: Train the model
    training_losses = []
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            predicted_state = model(batch.x, batch.edge_index, batch.global_features)
            loss = loss_fn(predicted_state, batch.target_state)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        training_losses.append(avg_loss)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}")

    # Step 5: Validate the model
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            predicted_state = model(batch.x, batch.edge_index, batch.global_features)
            loss = loss_fn(predicted_state, batch.target_state)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}")

    torch.save(model.state_dict(), "trained_gca_model.pth")
    print("Trained model saved as 'trained_gca_model.pth'")
    return model, training_losses, avg_val_loss

# Directories and parameter files
if __name__ == "__main__":
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    trained_model, training_losses, validation_loss = main(data_dirs, param_files)


Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/2.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/3.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/4.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/5.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/6.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/7.py
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/8.py
P

In [11]:
import os
import numpy as np
import torch
from torch_geometric.data import Data

# Define sequential prediction
def sequential_prediction(data_dir, param_file, model, start_timepoint, timepoints, output_dir):
    """
    Sequentially predict the state, property, and graph matrices starting from the initial timepoint.
    Args:
        data_dir (str): Directory containing the .npy files.
        param_file (str): Path to the parameter file.
        model (GCAModel): Trained GCA model.
        start_timepoint (int): Initial timepoint for prediction.
        timepoints (list): List of timepoints to predict.
        output_dir (str): Directory to save the predicted matrices.
    Returns:
        None
    """
    # Parse parameters and load initial matrices
    params = parse_parameters(param_file)
    global_constants = np.concatenate([
        np.array(params["v0"]),
        np.array([params["Dr"], params["kappa_A"], params["kappa_P"], params["a"], params["k"]]),
        params["W"].flatten(),
    ])
    graph_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_graph_mat.npy"))
    state_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_state_mat.npy"))
    properties_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_properties_mat.npy"))

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Save the initial matrices
    np.save(os.path.join(output_dir, f"state_prediction_{start_timepoint}.npy"), state_mat)
    np.save(os.path.join(output_dir, f"properties_prediction_{start_timepoint}.npy"), properties_mat)
    np.save(os.path.join(output_dir, f"graph_prediction_{start_timepoint}.npy"), graph_mat)
    print(f"Saved initial predictions for timepoint {start_timepoint}")

    # Iteratively predict subsequent timepoints
    for t in timepoints[1:]:
        node_features = np.hstack((properties_mat, np.argmax(state_mat, axis=1).reshape(-1, 1)))
        edge_indices = np.transpose(np.nonzero(graph_mat))
        graph_data = Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=torch.tensor(edge_indices.T, dtype=torch.long),
            global_features=torch.tensor(global_constants, dtype=torch.float),
        )

        # Predict the next state
        model.eval()
        with torch.no_grad():
            predicted_state = model(
                graph_data.x,
                graph_data.edge_index,
                graph_data.global_features,
            ).cpu().numpy()

        # Use the model output to update properties and graph predictions
        predicted_properties = predicted_state[:, :properties_mat.shape[1]]  # First columns as updated properties
        predicted_graph = graph_mat  # EDIT!!!!
        
        # Save the predicted matrices
        np.save(os.path.join(output_dir, f"{t}_state_mat.npy"), predicted_state)
        np.save(os.path.join(output_dir, f"{t}_properties_mat.npy"), predicted_properties)
        np.save(os.path.join(output_dir, f"{t}_graph_mat.npy"), predicted_graph)
        print(f"Saved predictions for timepoint {t}")

        # Update matrices for the next iteration
        state_mat = predicted_state
        properties_mat = predicted_properties
        graph_mat = predicted_graph

# Example usage
if __name__ == "__main__":
    # Assume trained_model is already loaded or trained
    data_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/matrix_output_1_test"
    param_file = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py"
    output_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/predictions"

    start_timepoint = 0
    timepoints = list(range(0, 2001, 100))  # Predict 0, 400, 800, ..., 2000

    sequential_prediction(data_dir, param_file, trained_model, start_timepoint, timepoints, output_dir)


Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Saved initial predictions for timepoint 0
Saved predictions for timepoint 100
Saved predictions for timepoint 200
Saved predictions for timepoint 300
Saved predictions for timepoint 400
Saved predictions for timepoint 500
Saved predictions for timepoint 600
Saved predictions for timepoint 700
Saved predictions for timepoint 800
Saved predictions for timepoint 900
Saved predictions for timepoint 1000
Saved predictions for timepoint 1100
Saved predictions for timepoint 1200
Saved predictions for timepoint 1300
Saved predictions for timepoint 1400
Saved predictions for timepoint 1500
Saved predictions for timepoint 1600
Saved predictions for timepoint 1700
Saved predictions for timepoint 1800
Saved predictions for timepoint 1900
Saved predictions for timepoint 2000
